# ValPAS prototype

### Dependencies
Dependencies to run this notebook should be installed according to the README.md located at the root folder of the project.
### Jupyter notebook "frontmatter"
The following lines of code are overhead to make loading of the valpas package possible from within the notebook without having to install the package via pip / setuptools. Essentially this loads the 'src/' folder into the sys path as searchable for modules. 

In [1]:
# iPython magic to autoreload modules everytime code is executed to propagate changes to the code
%load_ext autoreload
%autoreload 2

# loading '/src' into sys path
import os
import sys
module_path = os.path.abspath(os.path.join('..', 'src'))
if module_path not in sys.path:
    sys.path.append(module_path)

# don't show warnings
import warnings
warnings.filterwarnings('ignore')

### Defining the file path to the sample files:

In [2]:
# path to the csv containing the proteome data
fpath_prot = os.path.join(
    os.getcwd(),
    "..",
    "sample_data",
    "in_test_data",
    "rhodo_test_data_proteomics.csv"
    )

# path to the csv containing the metabolomics data
fpath_metabol = os.path.join(
    os.getcwd(),
    "..",
    "sample_data",
    "in_test_data",
    "rhodo_test_data_metabolomics.csv"
    )

## Association Examples
### Correlation between samples
Correlation can be calculated between instances within one type of data (e.g. protein-protein) or between two types of data (e.g. protein-metabolite). Below both variants are demonstrated (first within one type then across types). The returned data is a series of correlated instances sorted by the strenght of their correlation.

The ValPAS module offers a command line tool to perform all operations (see also `src/valpas/valpas.py`). We will be importing the `main` function from `valpas/valpas.py` and use this to execute commands in this Jupyter Notebook.

In [3]:
from valpas.valpas import main as valpas_main

#### Correlation between samples from one data type
Next we will calculate the correlation between instances within one data type. In this case the data type are protein abundances. The returned list is sorted descending (i.e. stronges correlation first).

The command issued below is equivalent to calling `python valpas.py associate -i INFILE` from the command line, where `INFILE='../sample_data/in_test_data/rhodo_test_data_proteomics.csv'` as defined in [Defining the file path...](#defining-the-file-path-to-the-sample-files).

If not further defined the script defaults to print the sorted list of id pairs with their association score to `stdout`.

In [4]:
%%capture out --no-stderr
# above line does some magic to capture the stdout
valpas_main(['associate', '-i', fpath_prot])

In [5]:
# some short code to display the first 10 entries printed to stdout 
count = 0
for line in out.stdout.split('\n'):
    print(line)
    count += 1
    if count > 10:
        break

RTO4_ID_1,RTO4_ID_2,Correlation
8369,8369,1.0
8370,8369,1.0
8372,8372,1.0
8369,8370,1.0
8370,8370,1.0
8371,8371,1.0
8373,8373,1.0
8371,8373,0.5685650527770536
8373,8371,0.5685650527770536
8373,8370,0.266250374168688


#### Correlation between samples from two different data types
Next we calculate the correlation between instances across two different types of data (in this example proteins and metabolites). Note that the function `calc_correlation` also accepts keyword parameters. If `fpath_2` is omitted as in the example above only the correlation between instances in the dataset designated by `fpath_1` is calculated. If both `fpath_1` and `fpath2` are present correlation between pairs of instances across both datasets are calculated. Additionally, note that the correlation function `corr_func` can be defined. Options are `'pearson'`, `'spearman'` and `'kendall'`. Note that if omitted the calculation defaults to `'pearson'`.

We can also further define an output path for the generated csv. In this case we will store the data to `'../sample_data/rhodo_multi_prot-metabol_corr.csv'` (defined by the parameter `-o OUTFILE`).

Again the equivalent command on the command line would be `python valpas.py associate -i INFILE -I INFILE2 -a spearman -o OUTFILE` where `INFILE='../sample_data/in_test_data/rhodo_test_data_proteomics.csv'` and `INFILE2='../sample_data/in_test_data/rhodo_test_data_metabolomics.csv'` as defined in [Defining the file path...](#defining-the-file-path-to-the-sample-files).



In [6]:
# defining the outfile path
out_fpath = os.path.join(
    os.getcwd(),
    "..",
    "sample_data",
    "out",
    "rhodo_test_prot-metabol_corr_list.csv"
    )

valpas_main([
    'associate',
    '-i', fpath_prot,
    '-I', fpath_metabol,
    '-a', 'spearman',
    '-o', out_fpath
    ])

#### Saving correlation matrices
It is also possible to print the resulting correlation matrix raw instead of outputting a sorted list of id pairs with their association score. This can be achieved using the `'-ot correlation_matrix'` option of the command line tool. Note if `'-ot'` is omitted it defaults to `'sorted_list'` and prints / stores said list. A example command is shown below.

The resulting CSV contains ids in the header as well as first column

In [7]:
out_fpath = os.path.join(
    os.getcwd(),
    "..",
    "sample_data",
    "out",
    "rhodo_test_prot-metabol_corr_mat.csv"
    )

valpas_main([
    'associate',
    '-i', fpath_prot,
    '-I', fpath_metabol,
    '-a', 'spearman',
    '-ot', 'correlation_matrix',
    '-o', out_fpath
    ])

### Other association types

Besides [Correlation between samples](#correlation-between-samples) there are also additional association types that can be calculated between samples. One of them is mutual information. In the this case the internal methods handle binning of the sample data values and calculates mutual information between samples.

Choosing mutual information as association type is done with the same command line option as correlation coefficients.

For a full list of association types see the output of `python valpas.yp associate -h`

In [8]:
%%capture out --no-stderr

valpas_main([
    'associate',
    '-i', fpath_prot,
    '-a', 'mutual_information',
    ])

In [9]:
count = 0
for line in out.stdout.rstrip('\n').split('\n'):
    print(line)
    count += 1
    if count > 10:
        break

print(*('Number of associations:', (len(out.stdout.rstrip('\n').split('\n'))-1)), sep=" ")

RTO4_ID_1,RTO4_ID_2,Correlation
8373,8372,1.6094379124341005
8372,8373,1.6094379124341005
8369,8369,1.0
8370,8370,1.0
8372,8372,1.0
8371,8371,1.0
8373,8373,1.0
8373,8371,0.6730116670092563
8371,8372,0.6730116670092563
8371,8373,0.6730116670092563
Number of associations: 25


### Additional options

Additionally the imported DataFrame can be filtered to exlude instances of interest from the association study based on their experimental completeness. E.g. if certain metabolites were only detected in a limited run of samples these metabolites can be excluded from analysis based on a user defined threshold.

This can be done by defining the command line argument `'-f / --filter_missing_values'`. The cutoff can be defined in the range of [0.0, 1.0]. By default (i.e. the command line option is omitted) no filtering will be done.

Below is an example call that includes the filtering step.

In [10]:
%%capture out

valpas_main([
    'associate',
    '-i', fpath_metabol,
    '-a', 'mutual_information',
    '-f', '0.5'
    ])

Note if instances are removed from consideration they are returned and printed to `stderr` (see below).

In [11]:
out.stderr.rstrip('\n')

'Removed items: protocatechuic acid, 2-hydroxycinnamic acid'

In [12]:
count = 0
for line in out.stdout.rstrip('\n').split('\n'):
    print(line)
    count += 1
    if count > 10:
        break

print(*('Number of associations:', (len(out.stdout.rstrip('\n').split('\n'))-1)), sep=" ")

Metabolite_1,Metabolite_2,Correlation
"1,3-dihydroxyacetone",2-hydroxyglutaric acid,1.945910149055313
2-hydroxyglutaric acid,"1,3-dihydroxyacetone",1.945910149055313
"1,3-dihydroxyacetone",2-furoic acid,1.7478680974667573
2-furoic acid,"1,3-dihydroxyacetone",1.7478680974667573
2-furoic acid,2-hydroxyglutaric acid,1.7478680974667573
2-hydroxyglutaric acid,2-furoic acid,1.7478680974667573
"1,3-dihydroxyacetone","1,3-dihydroxyacetone",1.0
2-furoic acid,2-furoic acid,1.0
2-hydroxyglutaric acid,2-hydroxyglutaric acid,1.0
Number of associations: 9


## Visualization

We can visualize some of the results using the `visualize` command of `valpas.py`. At this time only heatmap visualization is implemented and only in a very basic form.

Below is a example command to generate a heatmap visualization of the associations generated from the command line using the [previously generated correlation matrix](#saving-correlation-matrices) between portein identfiers and metabolites.

First we define some basic file paths and then execute the `valpas.py visualize` subroutine to create a heatmap of the imported correlation matrix.

Note that if executed from the command line the resulting plot will not be directly displayed. An outfile path for the graphic **has to be defined** otherwise the graphic will be lost. Executing the command in a Jupyter Notebook will also output the graphic into the notebook it self (the outfile path option does not have to be selected for this.)

In [13]:
fpath = os.path.join(
    os.getcwd(),
    "..",
    "sample_data",
    "out",
    "rhodo_test_prot-metabol_corr_mat.csv"
    )
fpath_fig = os.path.join(
    os.getcwd(),
    "..",
    "sample_data",
    "out",
    "rhodo_test_prot_corr_mat.png"
    )

valpas_main([
    'visualize',
    '-i', fpath,
    '-t', 'heatmap',
    '-c', 'Spearman Correlation',
    '-o', fpath_fig

])

usage: ipykernel_launcher.py visualize [-h] -i INFILE [-t {heatmap,graph}]
                                       [-cl LABEL] [-o OUTFILE]
ipykernel_launcher.py visualize: error: argument -i/--infile: can't open '/Users/mahl006/git_repos/valpas-prototype/notebooks/../sample_data/out/rhodo_test_prot_corr_mat.csv': [Errno 2] No such file or directory: '/Users/mahl006/git_repos/valpas-prototype/notebooks/../sample_data/out/rhodo_test_prot_corr_mat.csv'


AttributeError: 'tuple' object has no attribute 'tb_frame'